# Reconstruction benchmark — data & statistics explorer

What does the constraint-reconstruction benchmark actually feed the engines?
This notebook walks one dataset through the full pipeline, *without fitting
anything* (no JAX needed, runs on local CPU):

1. the raw table and its column types,
2. the shared discrete grid (`TableEncoder` — train-quantile bins for
   continuous columns, one bin per level for categoricals),
3. the oracle constraint bag (`true_bag` — all 1-way marginals + random
   pairwise tail probabilities, conditional probabilities, conditional
   expectations),
4. the degraded bag (`noisy_bag` — logit-scale jitter + deliberate conflicts),
5. how much pairwise dependence exists to recover, and which pairs the bag
   touches (`pair_tv_seen`) vs leaves to the inductive bias (`pair_tv_unseen`).

Change `DATASET` below and re-run all. First run of a real table fetches it
from OpenML/sklearn and caches locally.

In [ ]:
import sys, pathlib
# make `benchmarks` importable whether run from the repo root or benchmarks/
sys.path[:0] = [str(pathlib.Path.cwd()), str(pathlib.Path.cwd().parent)]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from benchmarks.datasets import DATASETS
from benchmarks.encoding import TableEncoder
from benchmarks.constraints import (true_bag, noisy_bag, MarginalConstraint,
                                    ProbConstraint, CondProbConstraint,
                                    CondExpectConstraint, _rows_in)
from benchmarks.metrics import _tv, _empirical_pair

DATASET = "adult"        # adult | bank_marketing | california | nursery | synthetic_chain
N_BINS_CONT = 16         # matches run.py default
BAG_SEED = 0
N_PAIR, N_COND, N_COND_EXPECT = 40, 20, 10   # run.py defaults

train_df, test_df = DATASETS[DATASET]()
print(f"{DATASET}: train {train_df.shape}, test {test_df.shape}")
train_df.head()

## 1. Columns and the shared grid

Every engine sees the same encoding: continuous columns are binned on
**train-split quantiles** (uniform in encoded space = quantile in raw space;
spike-heavy columns give the mode its own bin), categoricals get one bin per
level. `eff_bins = exp(entropy)` shows how evenly the mass spreads — a
16-bin column with eff_bins ≈ 3 is dominated by a few bins.

In [ ]:
enc = TableEncoder(train_df, n_bins_cont=N_BINS_CONT)
Xi = enc.bin_indices(train_df)
Xi_test = enc.bin_indices(test_df)

rows = []
for s in enc.specs:
    counts = np.bincount(Xi[:, enc.site[s.name]], minlength=s.n_bins)
    p = counts / counts.sum()
    ent = -np.sum(np.where(p > 0, p * np.log(p), 0.0))
    rows.append(dict(column=s.name, kind=s.kind, n_bins=s.n_bins,
                     eff_bins=round(float(np.exp(ent)), 2),
                     detail=(", ".join(s.levels[:4]) + (", ..." if len(s.levels) > 4 else "")
                             if s.kind == "cat" else
                             f"edges {s.edges[0]:.3g} .. {s.edges[-1]:.3g}")))
print(f"total grid size: {len(enc.names)} sites, dims {enc.dims}")
pd.DataFrame(rows)

### Encoded marginals

Bar chart per site, on the encoded grid. For continuous columns quantile
binning makes these near-uniform *by construction* (deviations = collapsed
duplicate edges / mode bins); categoricals keep their raw level frequencies.
These histograms are exactly the `MarginalConstraint` targets.

In [ ]:
ncol = 4
nrow = -(-len(enc.names) // ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 2.2 * nrow))
for ax, name in zip(axes.flat, enc.names):
    d = enc.dims[enc.site[name]]
    counts = np.bincount(Xi[:, enc.site[name]], minlength=d)
    ax.bar(np.arange(d), counts / counts.sum(), width=0.9)
    ax.set_title(f"{name} ({d})", fontsize=9)
    ax.set_xticks([])
for ax in axes.flat[len(enc.names):]:
    ax.axis("off")
fig.tight_layout()
plt.show()

## 2. The oracle constraint bag

`true_bag` computes, from the **train split only**:

- one `MarginalConstraint` per column (the full histogram above),
- `n_pair` random **pairwise tail probabilities**: P(X_a in tail, X_b in tail)
  for random one-sided tails,
- `n_cond` **conditional probabilities** P(tail_a | tail_b) (skipped when the
  conditioning event has < 30 train rows),
- `n_cond_expect` **conditional expectations** E[X_a | tail_b] in encoded
  [0,1] units (bin centres).

Below: the bag composition, then each constraint decoded back to raw space.

In [ ]:
def describe_interval(enc, iv):
    s = enc.specs[enc.site[iv.var]]
    if s.kind == "cat":
        lv = s.levels[iv.lo:iv.hi]
        body = "{" + ", ".join(lv[:3]) + (", ...}" if len(lv) > 3 else "}")
        return f"{iv.var} in {body}"
    lo, hi = s.edges[iv.lo], s.edges[iv.hi]
    if iv.lo == 0:
        return f"{iv.var} < {hi:.4g}"
    if iv.hi == s.n_bins:
        return f"{iv.var} >= {lo:.4g}"
    return f"{iv.var} in [{lo:.4g}, {hi:.4g})"


def describe(enc, c):
    if isinstance(c, MarginalConstraint):
        return f"marginal[{c.var}]  ({len(c.probs)} bins)"
    if isinstance(c, ProbConstraint):
        return "P(" + " & ".join(describe_interval(enc, e) for e in c.events) +                f") = {c.target:.4f}"
    if isinstance(c, CondProbConstraint):
        return "P(" + " & ".join(describe_interval(enc, e) for e in c.event) +                " | " + " & ".join(describe_interval(enc, e) for e in c.given) +                f") = {c.target:.4f}"
    if isinstance(c, CondExpectConstraint):
        return f"E[{c.var} | " +                " & ".join(describe_interval(enc, e) for e in c.given) +                f"] = {c.target:.4f}  (encoded units)"
    raise TypeError(type(c))


bag, constrained_pairs = true_bag(Xi, enc, seed=BAG_SEED, n_pair=N_PAIR,
                                  n_cond=N_COND, n_cond_expect=N_COND_EXPECT)
from collections import Counter
print(Counter(type(c).__name__ for c in bag))
print(f"{len(constrained_pairs)} distinct variable pairs touched\n")
for c in bag:
    if not isinstance(c, MarginalConstraint):
        print(describe(enc, c))

### Target distributions

Where do the targets live? Pairwise tail probabilities cluster near 0 (two
random tails rarely co-occur); conditional probabilities and expectations are
better spread. This matters for the noise model: logit-scale jitter moves a
p = 0.02 target much further in absolute terms than a p = 0.5 one is moved
relatively.

In [ ]:
groups = {"ProbConstraint": [], "CondProbConstraint": [], "CondExpectConstraint": []}
for c in bag:
    if type(c).__name__ in groups:
        groups[type(c).__name__].append(c.target)
fig, axes = plt.subplots(1, 3, figsize=(11, 2.6))
for ax, (name, ts) in zip(axes, groups.items()):
    ax.hist(ts, bins=20, range=(0, 1))
    ax.set_title(f"{name} (n={len(ts)})", fontsize=9)
fig.tight_layout()
plt.show()

## 3. Degrading the bag: `noisy_bag`

The sweep's realism axis. Probabilities are jittered on the **log-odds**
scale (LLM estimates live around logit-sd 0.3–1.0), expectations get Gaussian
noise in encoded units, and `conflict_frac` replaces that fraction of
non-marginal constraints with uniform-random garbage. Left: oracle vs noisy
targets at increasing logit-sd. Right: a 20% conflict draw — the off-diagonal
points are the deliberately wrong targets the robustness machinery must
absorb.

In [ ]:
oracle_t = np.array([c.target for c in bag if not isinstance(c, MarginalConstraint)])

fig, axes = plt.subplots(1, 4, figsize=(13, 3.1))
for ax, sd in zip(axes[:3], [0.1, 0.3, 1.0]):
    nb = noisy_bag(bag, seed=1, prob_logit_sd=sd, expect_sd=sd * 0.1)
    noisy_t = np.array([c.target for c in nb if not isinstance(c, MarginalConstraint)])
    ax.scatter(oracle_t, noisy_t, s=12, alpha=0.7)
    ax.plot([0, 1], [0, 1], "k--", lw=0.7)
    ax.set_title(f"logit-sd = {sd}", fontsize=9)
    ax.set_xlabel("oracle target")
axes[0].set_ylabel("noisy target")

nb = noisy_bag(bag, seed=1, prob_logit_sd=0.3, expect_sd=0.03, conflict_frac=0.2)
noisy_t = np.array([c.target for c in nb if not isinstance(c, MarginalConstraint)])
conf = np.abs(noisy_t - oracle_t) > 0.15
axes[3].scatter(oracle_t[~conf], noisy_t[~conf], s=12, alpha=0.7, label="jittered")
axes[3].scatter(oracle_t[conf], noisy_t[conf], s=16, color="crimson", label="conflicted?")
axes[3].plot([0, 1], [0, 1], "k--", lw=0.7)
axes[3].set_title("logit-sd 0.3 + conflict 0.2", fontsize=9)
axes[3].legend(fontsize=7)
fig.tight_layout()
plt.show()

## 4. How much dependence is there to recover?

Per-pair TV distance between the empirical **train** pairwise marginal and the
product of its 1-way marginals — i.e. how wrong the `independent` null is on
each pair. Dark cells = correlations an engine can exploit. The dots mark
pairs the bag touched: `pair_tv_seen` averages over dotted cells,
`pair_tv_unseen` over the rest — the unseen score is pure inductive bias.

In [ ]:
D = len(enc.names)
dep = np.full((D, D), np.nan)
for i in range(D):
    for j in range(i + 1, D):
        a, b = enc.names[i], enc.names[j]
        joint = _empirical_pair(Xi, enc, a, b)
        pa, pb = joint.sum(1), joint.sum(0)
        dep[i, j] = dep[j, i] = _tv(joint, np.outer(pa, pb))

fig, ax = plt.subplots(figsize=(0.55 * D + 2, 0.55 * D + 1))
im = ax.imshow(dep, cmap="viridis")
ax.set_xticks(range(D)); ax.set_xticklabels(enc.names, rotation=90, fontsize=7)
ax.set_yticks(range(D)); ax.set_yticklabels(enc.names, fontsize=7)
for pr in constrained_pairs:
    a, b = sorted(pr)
    i, j = enc.site[a], enc.site[b]
    ax.plot([j, i], [i, j], "o", color="white", ms=3, mec="black", mew=0.4)
fig.colorbar(im, label="TV(joint, product of marginals)")
ax.set_title("train-split pairwise dependence; dots = pairs touched by the bag", fontsize=9)
fig.tight_layout()
plt.show()

seen = [dep[enc.site[a], enc.site[b]] for a, b in (sorted(p) for p in constrained_pairs)]
unseen = [dep[i, j] for i in range(D) for j in range(i + 1, D)
          if frozenset((enc.names[i], enc.names[j])) not in constrained_pairs]
print(f"mean dependence TV  seen: {np.mean(seen):.4f}   unseen: {np.mean(unseen):.4f}"
      f"   (pairs: {len(seen)} seen / {len(unseen)} unseen)")

## 5. Train vs test drift

Sanity floor for the fit metrics: even a perfect engine can't score below the
train/test sampling gap. Per-column marginal TV and, for the constrained
statistics, the oracle target recomputed on the test split.

In [ ]:
tvs = []
for name in enc.names:
    d = enc.dims[enc.site[name]]
    ptr = np.bincount(Xi[:, enc.site[name]], minlength=d)
    pte = np.bincount(Xi_test[:, enc.site[name]], minlength=d)
    tvs.append(_tv(ptr / ptr.sum(), pte / pte.sum()))
print(f"train/test marginal TV: mean {np.mean(tvs):.4f}, max {np.max(tvs):.4f}")

drift = []
for c in bag:
    if isinstance(c, ProbConstraint):
        drift.append(abs(c.target - float(_rows_in(Xi_test, enc, c.events).mean())))
print(f"pairwise-prob target train/test drift: mean {np.mean(drift):.4f}, "
      f"max {np.max(drift):.4f}  (n={len(drift)})")